In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import roc_auc_score, f1_score, classification_report
import category_encoders as ce

In [20]:
train = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
df = train.merge(identity, how='left', on='TransactionID')
from sklearn.model_selection import train_test_split

X = df.drop(columns=["isFraud"])
y = df["isFraud"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Feature Engineering

In [ ]:
class InfCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X = X.replace([np.inf, -np.inf], np.nan)
        return X

class DropHighNaN(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = X.columns[X.isna().mean() > self.threshold]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

class DropIDColumns(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.cols_to_drop_ = [c for c in X.columns if "id" in c.lower()]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")

class DropNearConstant(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.99):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.cols_to_drop_ = [
            col for col in X.columns
            if X[col].value_counts(normalize=True, dropna=False).iloc[0] > self.threshold
        ]
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_to_drop_, errors="ignore")



class LogSkewTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=1.0):
        self.threshold = threshold

    def fit(self, X, y=None):
        num = X.select_dtypes(include=np.number)
        self.skew_cols_ = num.columns[num.skew().abs() > self.threshold]
        return self

    def transform(self, X):
        X = X.copy()

        for col in self.skew_cols_:
            # replace inf just in case
            X[col] = X[col].replace([np.inf, -np.inf], np.nan)

            # fill NaN with 0 (safe baseline)
            X[col] = X[col].fillna(0)

            # handle negative values safely
            min_val = X[col].min()
            if min_val < 0:
                X[col] = X[col] - min_val  # shift to make all >= 0

            # now safe log transform
            X[col] = np.log1p(X[col])

        return X

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.rare_maps_ = {}

        cat_cols = X.select_dtypes(include="object").columns

        for col in cat_cols:
            freq = X[col].value_counts(normalize=True)
            self.rare_maps_[col] = freq[freq < self.threshold].index

        return self

    def transform(self, X):
        X = X.copy()

        for col, rare_vals in self.rare_maps_.items():
            X[col] = X[col].replace(rare_vals, "Other")

        return X



In [9]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
import numpy as np

class ThresholdClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator
        self.threshold = threshold

    def fit(self, X, y):
        self.model_ = clone(self.estimator)
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        probs = self.model_.predict_proba(X)[:, 1]
        return (probs >= self.threshold).astype(int)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

# Pipeline

In [ ]:
import category_encoders as ce
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OrdinalEncoder

pre_cleaning = Pipeline(steps=[
    ("inf_clean", InfCleaner()),
    ("drop_nan", DropHighNaN()),
    ("drop_id", DropIDColumns()),
    ("drop_const", DropNearConstant()),
    ("log_skew", LogSkewTransformer()),
    ("rare_groups", RareCategoryGrouper()),
])


X_clean = pre_cleaning.fit_transform(X_train)


num_cols = X_clean.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_clean.select_dtypes(include=["object", "category"]).columns.tolist()


numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])


categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])


feature_pipeline = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)


model = Pipeline(steps=[
    ("pre_clean", pre_cleaning),
    ("features", feature_pipeline),

    ("clf", ThresholdClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=8,
            min_samples_split=20,
            min_samples_leaf=10,
            criterion="gini",
            class_weight="balanced",
            random_state=42
        ),
        threshold=0.9334319666643047
    ))
])



with class_weight={0: 0.56, 1: 5.0},

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "clf__max_depth": [3, 5, 10, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__criterion": ["gini", "entropy"]
}

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=15,  
    cv=3,       
    scoring="recall",
    n_jobs=1,   
    verbose=2,
    random_state=42
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_

print("Best Params:", random_search.best_params_)
print("Best Recall:", random_search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


ValueError: Invalid parameter 'min_samples_split' for estimator ThresholdClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                     max_depth=8,
                                                     min_samples_leaf=10,
                                                     min_samples_split=20,
                                                     random_state=42)). Valid parameters are: ['estimator', 'threshold'].

In [28]:
model.fit(X_train, y_train)

Pipeline(steps=[('pre_clean',
                 Pipeline(steps=[('inf_clean', InfCleaner()),
                                 ('drop_nan', DropHighNaN()),
                                 ('drop_id', DropIDColumns()),
                                 ('drop_const', DropNearConstant()),
                                 ('log_skew', LogSkewTransformer()),
                                 ('rare_groups', RareCategoryGrouper())])),
                ('features',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),...
                                                                   OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                                  unknown_value=-1))]),
                                                  ['ProductCD', 'card4',
                                                   'card6', 'P_emaildomain',
                                                   'R_emaildomain', 'M1', 'M2',
                                                   'M3', 'M4', 'M5', 'M6', 'M7',
                                                   'M8', 'M9', 'DeviceType',
                                                   'DeviceInfo'])])),
                ('clf',
                 ThresholdClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                                      max_depth=8,
                                                                      min_samples_leaf=10,
                                                                      min_samples_split=20,
                                                                      random_state=42),
                                     threshold=0.9334319666643047))])

# Overfitting

best model that I choosed in increase f1

In [12]:
y_train_pred = model.predict(X_train)
print(classification_report(y_train_pred, y_train))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00    446025
           1       1.00      1.00      1.00     26407

    accuracy                           1.00    472432
   macro avg       1.00      1.00      1.00    472432
weighted avg       1.00      1.00      1.00    472432



In [13]:
y_pred = model.predict(X_valid)
print(classification_report(y_valid, y_pred))


              precision    recall  f1-score   support

           0       0.99      0.96      0.97    113975
           1       0.35      0.61      0.45      4133

    accuracy                           0.95    118108
   macro avg       0.67      0.78      0.71    118108
weighted avg       0.96      0.95      0.95    118108



try something with overfitting

In [ ]:
model = Pipeline(steps=[
    ("pre_clean", pre_cleaning),
    ("features", feature_pipeline),

    ("clf", ThresholdClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=8,
            min_samples_split=20,
            min_samples_leaf=10,
            criterion="gini",
            class_weight="balanced",
            random_state=42
        )#,
        #threshold=0.6666666666666666
    ))
])

In [24]:
y_train_pred = model.predict(X_train)
print(classification_report(y_train_pred, y_train))


              precision    recall  f1-score   support

           0       0.78      0.99      0.87    358752
           1       0.79      0.11      0.20    113680

    accuracy                           0.78    472432
   macro avg       0.78      0.55      0.54    472432
weighted avg       0.78      0.78      0.71    472432



In [29]:
y_train_pred = model.predict(X_train)
print(classification_report(y_train_pred, y_train))
# added trashold

              precision    recall  f1-score   support

           0       0.99      0.98      0.98    462008
           1       0.36      0.58      0.45     10424

    accuracy                           0.97    472432
   macro avg       0.68      0.78      0.72    472432
weighted avg       0.98      0.97      0.97    472432



In [30]:
y_valid_pred = model.predict(X_valid)
print(classification_report(y_valid_pred, y_valid))
# added trashold

              precision    recall  f1-score   support

           0       0.99      0.98      0.98    115469
           1       0.36      0.56      0.44      2639

    accuracy                           0.97    118108
   macro avg       0.67      0.77      0.71    118108
weighted avg       0.98      0.97      0.97    118108



#

# increate f1

In [18]:
y_pred = model.predict(X_valid)
print(classification_report(y_valid, y_pred))
# without weight

              precision    recall  f1-score   support

           0       0.98      0.99      0.99    113975
           1       0.65      0.52      0.58      4133

    accuracy                           0.97    118108
   macro avg       0.82      0.75      0.78    118108
weighted avg       0.97      0.97      0.97    118108



In [28]:
y_pred = model.predict(X_valid)
print(classification_report(y_valid, y_pred))

# with weight


              precision    recall  f1-score   support

           0       0.99      0.95      0.97    113975
           1       0.32      0.64      0.43      4133

    accuracy                           0.94    118108
   macro avg       0.65      0.79      0.70    118108
weighted avg       0.96      0.94      0.95    118108



In [26]:
from sklearn.metrics import precision_recall_curve

y_proba = model.predict_proba(X_valid)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_valid, y_proba)

f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)

best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

Best threshold: 0.9334319666643047


In [23]:
y_pred = (y_proba >= best_threshold).astype(int)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.99    113975
           1       0.75      0.48      0.59      4133

    accuracy                           0.98    118108
   macro avg       0.87      0.74      0.79    118108
weighted avg       0.97      0.98      0.97    118108



without class weight

In [25]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
# Hyperparameter grid
param_grid = {
    "clf__max_depth": [3, 5, 10, None],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 5],
    "clf__criterion": ["gini", "entropy"]
}

# Grid Search
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=15,   # instead of 72
    cv=3,        # less memory than 5
    scoring="recall",
    n_jobs=1,    # important for Kaggle RAM
    verbose=2,
    random_state=42
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_

print("Best Params:", random_search.best_params_)
print("Best Recall:", random_search.best_score_)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
[CV] END clf__criterion=gini, clf__max_depth=3, clf__min_samples_leaf=2, clf__min_samples_split=5; total time= 1.2min
[CV] END clf__criterion=gini, clf__max_depth=3, clf__min_samples_leaf=2, clf__min_samples_split=5; total time= 1.2min
[CV] END clf__criterion=gini, clf__max_depth=3, clf__min_samples_leaf=2, clf__min_samples_split=5; total time= 1.2min
[CV] END clf__criterion=entropy, clf__max_depth=10, clf__min_samples_leaf=5, clf__min_samples_split=10; total time= 1.4min
[CV] END clf__criterion=entropy, clf__max_depth=10, clf__min_samples_leaf=5, clf__min_samples_split=10; total time= 1.4min
[CV] END clf__criterion=entropy, clf__max_depth=10, clf__min_samples_leaf=5, clf__min_samples_split=10; total time= 1.4min
[CV] END clf__criterion=gini, clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time= 1.5min
[CV] END clf__criterion=gini, clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=